# Recuperação de sessões pós-quânticas em IoT

Este notebook apresenta os resultados de referência e permite reproduzir os experimentos da pesquisa. A implementação está nas classes do simulador; as células configuram e chamam essas classes.

O `Tutorial_Execucao.ipynb` continua sendo o exemplo do simulador geral e não foi alterado.

## 1. Ambiente de execução

Para **consultar as tabelas de referência**, basta o ambiente Python do simulador. Para **repetir os experimentos TLS e Mininet-WiFi**, construa as imagens na raiz do repositório:

```bash
docker compose build notebook
docker build -t pqc-iot-tls:organizacao -f docker/tls/Dockerfile .
docker build -t pqc-iot-continuity:organizacao -f docker/continuity/Dockerfile .
docker compose -f docker-compose.yml -f docker/context_costs/compose.yml build notebook
docker compose -f docker-compose.yml -f docker/context_costs/compose.yml run --rm --service-ports notebook
```

Abra este notebook no kernel **Python 3 - PQC IoT Simulator**. Execute as células na ordem apresentada. O contêiner da pesquisa inclui OpenSSL 3.5.9 e os módulos TLS nativos. A execução de rede exige Linux, privilégios no contêiner e suporte a `mac80211_hwsim` no host.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Image
from pqc_iot_simulator import (
    RecoveryStudy, ReferenceResults, ValidationSuite,
)

referencia = ReferenceResults()
referencia.verify()

## 2. Configurações comparadas

Todos os perfis usam TLS 1.3, autenticação mútua, AES-256-GCM/SHA-384 e 0-RTT desabilitado.

| Perfil | Estabelecimento de chaves | Autenticação |
|---|---|---|
| `classical` | X25519 | ECDSA P-256 |
| `hybrid` | X25519MLKEM768 | ECDSA P-256 |
| `pqc` | X25519MLKEM768 | ML-DSA-44 |

Os tratamentos são `full` (autenticação completa direta), `resume_volatile` (chave dos tickets perdida no reinício) e `resume_preserved` (chave preservada). As duas políticas de retomada usam a mesma construção de tickets. O banco de leituras é preservado nessa comparação.

## 3. Resultados de referência: transmissão da raiz

O experimento compara transmissão e omissão da CA raiz **nos dois participantes**, mantendo a mesma raiz confiável. São 540 execuções, com dez blocos e falhas em 0%, 25% ou 100% das leituras elegíveis. Cada execução contém nove leituras, com payload de 31 bytes.

As tabelas abaixo conservam os custos absolutos e as comparações realizadas dentro da mesma repetição. Elas são dados versionados, não resultados de uma nova execução deste notebook.

In [ ]:
cadeias = referencia.table("chains")
display(cadeias.loc[
    (cadeias["profile"] == "pqc") & (cadeias["restart_fraction"] == 1),
    ["include_root", "treatment", "blocks", "median_ipv4_bytes_per_confirmed_reading"],
])
display(referencia.table("chain_comparisons").query("profile == 'pqc' and restart_fraction == 1"))

## 4. Resultados de referência: leituras por conexão

O experimento mantém 100 leituras de 512 bytes e limita cada conexão a 1, 10 ou 100 leituras. Compara ausência de falhas com reinício antes da 51ª leitura e supressão de sua primeira confirmação. A raiz não é transmitida, mas permanece confiável. São 540 execuções em dez blocos.

In [ ]:
amortizacao = referencia.table("amortization")
display(amortizacao.query("profile == 'pqc'")[[
    "recovery", "batch_size", "treatment", "blocks",
    "median_ipv4_bytes_per_confirmed_reading", "median_ms_per_reading",
]])

## 5. Repetir os experimentos

Mantenha `REPRODUZIR = False` para consultar apenas os resultados de referência. Altere para `True` para executar os experimentos no contêiner de pesquisa. Com dez blocos, a execução completa pode levar várias horas.

Os novos dados são gravados em um diretório exclusivo. JSON, configurações, agendas de falhas, capturas PCAP e verificações são preservados. Os resultados de referência em `docs/` não são substituídos. Bytes podem variar com a rede e tempos dependem do host; a reprodução verifica as mesmas condições e os mesmos critérios, sem exigir valores temporais idênticos.

`BLOCOS = 10` corresponde à pesquisa. Um valor menor permite conferir o funcionamento, mas representa outra quantidade de repetições. Use pelo menos dois blocos para calcular os intervalos da análise de continuidade; um bloco serve apenas para verificação funcional das etapas individuais.

In [ ]:
REPRODUZIR = False
BLOCOS = 10
identificador = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
estudo = RecoveryStudy(Path("results/reproducao") / identificador, blocks=BLOCOS)

if REPRODUZIR:
    display(estudo.preflight())
else:
    print("Consulta aos resultados de referência. Nenhum experimento novo foi executado.")

### 5.1 Verificações funcionais

Incluem restauração de estado, deduplicação, análise das comparações e cobertura de captura. Essas verificações permanecem dentro do pacote e são chamadas pela classe `ValidationSuite`.

In [ ]:
if REPRODUZIR:
    display(ValidationSuite().run())

### 5.2 Protótipo: chaves, contadores e interrupções

Executa as seis políticas e quatro falhas com dez repetições (240 execuções), seguidas dos onze cenários de interrupção e três modos com chaves novas (330 casos). O protótipo usa primitivas criptográficas reais e transporte lógico; seus bytes são distintos dos bytes TLS.

In [ ]:
if REPRODUZIR:
    display(estudo.run_prototype())

### 5.3 TLS: preservação do contexto e frequência de falhas

Executa 1.620 casos da matriz principal e 180 casos adicionais sob perdas de 1% e 5%, quando `BLOCOS = 10`. Conserva as agendas entre os comparadores, a ordem aleatorizada, os controles de duplicação e as tentativas até confirmação. Use pelo menos dois blocos para calcular os intervalos da análise de continuidade; um bloco serve apenas para verificação funcional das etapas individuais.

In [ ]:
if REPRODUZIR:
    display(estudo.run_continuity())

### 5.4 TLS: cadeia com e sem raiz

Repete a comparação dos três tratamentos e perfis, modificando apenas a transmissão da raiz. As cadeias efetivamente enviadas pelo cliente e pelo servidor são verificadas.

In [ ]:
if REPRODUZIR:
    display(estudo.run_chains())

### 5.5 TLS: amortização do handshake

Repete as cargas de 100 leituras com conexões de 1, 10 ou 100 leituras, com e sem recuperação. A captura inclui as tentativas e os encerramentos observados no servidor.

In [ ]:
if REPRODUZIR:
    display(estudo.run_amortization())

### 5.6 Tickets, identidade, persistência e checkpoint

Registra comprimentos públicos de certificados, sessões e tickets, incluindo um controle que altera somente a credencial do cliente. Verifica o vínculo entre o dispositivo e a identidade autenticada e interrompe a gravação antes/depois do commit e antes da confirmação. Diferencia encerramento com limpeza e término abrupto do processo; esses testes não representam queda física de energia.

In [ ]:
if REPRODUZIR:
    display(estudo.run_security())

## 6. Analisar e verificar os novos resultados

A análise conserva o bloco como unidade de repetição e calcula as diferenças dentro da mesma repetição. Não descarta leituras sem confirmação. A verificação confronta configurações, resultados, cadeias e capturas antes da exportação.

In [ ]:
if REPRODUZIR:
    display(estudo.analyze_continuity())
    display(estudo.analyze_context())
    display(referencia.compare(estudo.directory / "context_analysis"))

## 7. Gerar e consultar as figuras

As figuras usam as tabelas analisadas e o diagnóstico de tickets. Os gráficos permanecem coloridos. No modo de consulta, são mostradas as figuras de referência; ao reproduzir, são geradas figuras com os dados novos.

In [ ]:
if REPRODUZIR:
    pasta_figuras = estudo.export_figures()
else:
    pasta_figuras = Path("docs/figuras/contexto/artigo")

for nome in ("contexto_custos_sem_raiz", "contexto_economia_cadeias", "contexto_ticket_identidade", "contexto_leituras_custos", "contexto_leituras_recuperacao", "contexto_leituras_tempo"):
    arquivo = pasta_figuras / f"{nome}.png"
    display(Image(filename=str(arquivo), width=720))

## 8. Arquivos e reprodução

- **Implementação:** `pqc_iot_simulator/pqc_iot_sim/research/`.
- **Referência:** `docs/resultados_contexto_20261106/` e `docs/resultados_continuidade_20261021_v5/`.
- **Resultados novos:** o diretório escolhido em `estudo.directory`, com dados brutos, capturas e análises.
- **Explicação dos resultados:** `docs/resultado_contexto_20261106.md`.
- **Guia da organização:** `docs/reproducao/README.md`.

Os dados brutos gerados localmente não são incluídos automaticamente no Git. Para compartilhar uma nova execução, compartilhe seu diretório completo, incluindo manifestos e PCAP. Os checkpoints publicados preservam a implementação anterior à organização.